
# Capstone Project: Predicting High-Performing Firms

This notebook develops a machine learning classification project to identify high-performing firms using financial ratios and company characteristics.

The workflow includes:

1. Loading and preparing the financial dataset  
2. Creating financial ratios and the target variable  
3. Exploratory Data Analysis (EDA)  
4. Statistical hypothesis testing  
5. Train-test split and preprocessing  
6. Model training and evaluation  
7. Feature interpretation and business conclusions  



## Preprocessing

This section prepares the features for machine learning.

The preprocessing includes:

- imputing missing numerical values using the median  
- handling missing categorical values  
- encoding categorical variables  
- scaling numerical variables  

This ensures that models such as Logistic Regression and SVM can work correctly with the data.



## Baseline Model

A Dummy Classifier is used as a baseline model.

The purpose of the baseline is to check whether the machine learning models perform better than a simple model that predicts the majority class.



## Model Evaluation

Models are evaluated using:

- Accuracy: overall percentage of correct predictions  
- Precision: how reliable positive predictions are  
- Recall: how many actual high-performing firms were correctly identified  

Recall is especially important in this project because missing a truly high-performing firm may represent a lost investment opportunity.


In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import classification_report

## Data Preparation Before Exploratory Data Analysis (EDA)

* The project started with a raw financial dataset containing firm-level accounting variables and company characteristics.

* Initial preprocessing was conducted to understand the structure and quality of the dataset, including:

  * checking dataset dimensions,
  * identifying numerical and categorical variables,
  * inspecting data types,
  * and detecting missing values.

* Potentially irrelevant and leakage-related variables were removed to avoid using information that could artificially improve model performance.

* Feature engineering was then performed to transform raw accounting values into more meaningful financial indicators. Several financial ratios were created, including:

  * **Leverage ratio** to measure financial risk,
  * **Cash ratio** to capture liquidity,
  * **PPE ratio** to evaluate investment in fixed assets.

* A categorical firm size variable was also prepared for later analysis and encoding.

* A binary target variable called `high_performance` was created to classify firms into high-performing and low-performing groups.

* At this stage, missing values were identified and analyzed but not yet imputed, as imputation was intentionally performed after the train-test split to avoid data leakage during machine learning preprocessing.

* Following feature engineering and dataset restructuring, the resulting analytical dataset consisted primarily of engineered financial ratios, firm characteristics, and the target classification variable.

* This processed dataset was then used for Exploratory Data Analysis (EDA) to investigate distributions, outliers, correlations, and relationships between financial indicators and firm performance.


In [ ]:
df=pd.read_csv('final_dataset.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df['size'] = pd.qcut(
    df['Assets'], 
    q=3, 
    labels=['Small', 'Medium', 'Large']
)


In [ ]:
df.head()

In [ ]:
df['Liabilities'] = df['Assets'] - df['StockholdersEquity']
df['leverage'] = df['Liabilities'] / df['Assets'].replace(0, 1)

In [ ]:
df['risk'] = pd.qcut(
    df['leverage'],
    q=3,
    labels=['Low','Medium','High'],
    duplicates='drop'
)

In [ ]:
df.head()

In [ ]:
df['Liabilities'] = df['Assets'] - df['StockholdersEquity']
df['ROA'] = df['NetIncomeLoss'] / df['Assets'].replace(0, 1)
df['ppe_ratio'] = df['PropertyPlantAndEquipmentNet'] / df['Assets'].replace(0, 1)

df['equity_ratio'] = df['StockholdersEquity'] / df['Assets'].replace(0, 1)
df['cash_ratio'] = df['CashAndCashEquivalentsAtCarryingValue'] / df['Assets'].replace(0, 1)

df['retained_ratio'] = df['RetainedEarningsAccumulatedDeficit'] / df['Assets'].replace(0, 1)
df['high_performance'] = (df['ROA'] > df['ROA'].median()).astype(int)










In [ ]:
df=pd.read_csv('capst.csv')
dforiginal=df.copy()
df.head()


## Key EDA Findings

* Financial ratios were analyzed after feature engineering to improve comparability across firms.

* Company size showed the strongest visible relationship with firm performance.

* Leverage, cash ratio, and PPE ratio differed significantly between high-performing and low-performing firms.

* Cash ratio and leverage appeared to have the strongest statistical relationship with performance.

* Financial data showed skewness and outliers; therefore, non-parametric statistical tests were used.

* Mann–Whitney U tests confirmed statistically significant differences for:

  * leverage
  * cash ratio
  * PPE ratio

* A Chi-square test showed a very strong association between company size and firm performance.

* Overall, the analysis suggests that both firm size and financial ratios contribute to predicting high-performing firms.




## Exploratory Data Analysis (EDA)

This section explores the dataset before modelling.

The aim of EDA is to understand:

- missing values and data quality  
- distributions of financial variables  
- outliers and skewness  
- relationships between features  
- early patterns linked to firm performance  

These insights support feature selection and modelling decisions.


In [ ]:
df.isna().mean() *100

In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.head()

In [ ]:
#Missing values
(df.isna().mean()*100).sort_values(ascending=False)


In [ ]:
#TARGET BALANCE
df['high_performance'].value_counts(normalize=True)*100

In [ ]:
#CORRELATIONS HEATMAP
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(14,10))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.savefig("heatmap.png", bbox_inches='tight')
plt.show()

In [ ]:
plt.savefig("heatmap.png", bbox_inches='tight')

In [ ]:

###Compare features by target
sns.boxplot(x='high_performance', y='leverage', data=df)
plt.show()

In [ ]:
sns.boxplot(x='high_performance', y='cash_ratio', data=df)
plt.show()

In [ ]:
sns.boxplot(x='high_performance', y='ppe_ratio', data=df)
plt.show()

In [ ]:
## ppte 2 
filtered_df = df[df['ppe_ratio'] < df['ppe_ratio'].quantile(0.99)]

plt.hist(filtered_df['ppe_ratio'], bins=30)
plt.show()

In [ ]:
##categorical analysis
sns.countplot(x='size', hue='high_performance', data=df)
plt.show()

In [ ]:
#sns.histplot(data=df, x='leverage', hue='high_performance', kde=True)
#plt.show()

In [ ]:
small_df = df.sample(500, random_state=42)

plt.hist(
    small_df['leverage'],
    bins=30
)

plt.xlabel('Leverage')
plt.ylabel('Frequency')
plt.title('Distribution of Leverage')

plt.show()

In [ ]:
small_df = df.sample(500, random_state=42)

plt.hist(
    small_df[small_df['high_performance']==0]['leverage'],
    bins=30,
    alpha=0.5,
    label='Low Performance'
)

plt.hist(
    small_df[small_df['high_performance']==1]['leverage'],
    bins=30,
    alpha=0.5,
    label='High Performance'
)

plt.legend()
plt.xlabel('Leverage')
plt.ylabel('Frequency')

plt.show()


## Statistical Hypothesis Testing

This section tests whether key financial features differ significantly between high-performing and low-performing firms.

Because financial ratios are skewed and contain outliers, non-parametric tests such as the Mann–Whitney U test are suitable for continuous variables.

For categorical variables such as company size, the Chi-square test is used to assess whether there is an association with firm performance.


In [ ]:

##hypotesys test
from scipy.stats import mannwhitneyu

group0 = df[df['high_performance']==0]['leverage'].dropna()
group1 = df[df['high_performance']==1]['leverage'].dropna()

stat, p_value = mannwhitneyu(group0, group1)

print("P-value:", p_value)

In [ ]:
## lavarege differ fro both groups h0 rejected

In [ ]:
group0 = df[df['high_performance']==0]['cash_ratio'].dropna()
group1 = df[df['high_performance']==1]['cash_ratio'].dropna()

stat, p_value = mannwhitneyu(group0, group1)

print("P-value:", p_value)

In [ ]:
## cashratio differ fro both groups h0 rejected  
#Cash ratio differs significantly between
#high-performing and low-performing firms.

In [ ]:
group0 = df[df['high_performance']==0]['ppe_ratio'].dropna()
group1 = df[df['high_performance']==1]['ppe_ratio'].dropna()

stat, p_value = mannwhitneyu(group0, group1)

print("P-value:", p_value)

In [ ]:
#PPE ratio differs significantly between
#high-performing and low-performing firms.

In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['size'], df['high_performance'])

chi2, p, dof, expected = chi2_contingency(contingency)

print("P-value:", p)

In [ ]:
#Company size and firm performance are significantly associated.

## Feature Selection Process

* Feature selection was performed using a combination of:

  * correlation analysis,
  * domain knowledge from financial analysis,
  * interpretability considerations,
  * and data leakage prevention.

* A correlation matrix was used to identify highly correlated variables and reduce redundancy within the dataset. Features providing overlapping information were carefully reviewed to avoid multicollinearity and improve model stability.

* Financial domain knowledge was also applied to prioritize variables that are commonly associated with firm performance, such as leverage, liquidity, and investment structure ratios.

* Several engineered financial ratios were derived directly from raw accounting variables. Once these ratios were created, many of the original raw columns were removed because:

  * they contained overlapping information,
  * they could introduce redundancy,
  * and the ratios provided more standardized and interpretable measures across firms of different sizes.

* Potential leakage-related variables were excluded to ensure that the models only used information available prior to the target outcome.

* Preference was given to features that were:

  * economically meaningful,
  * interpretable,
  * and useful for both statistical analysis and machine learning modeling.

* The final selected features aimed to balance predictive performance with interpretability and robustness.


In [ ]:

features = ['leverage','cash_ratio','ppe_ratio','size']
target =[ 'high_performance']

X = df[features].copy()
y = df[target]







In [ ]:
X.select_dtypes(include='number').corr()

In [ ]:
print(type(X))

In [ ]:
df['high_performance'].value_counts(normalize=True)


## Train-Test Split

The dataset is split into training and test sets before final preprocessing and modelling.

This helps evaluate how well the models generalise to unseen data.

Preprocessing steps such as imputation, scaling and encoding should be learned from the training data only to reduce the risk of data leakage.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
categorical_cols = ['size']
numerical_cols = ['leverage','cash_ratio','ppe_ratio']
X_train['size'] = X_train['size'].astype(str)
X_test['size'] = X_test['size'].astype(str)

X_train['size'] = X_train['size'].replace('nan', 'Unknown')
X_test['size'] = X_test['size'].replace('nan', 'Unknown')
num_cols = X_train.select_dtypes(include='number').columns
cat_cols = X_train.select_dtypes(exclude='number').columns
# compute median on train only
median = X_train[num_cols].median()
X_train[num_cols] = X_train[num_cols].fillna(median)
X_test[num_cols] = X_test[num_cols].fillna(median)
X_train[cat_cols] = X_train[cat_cols].fillna('Unknown')
X_test[cat_cols] = X_test[cat_cols].fillna('Unknown')



In [ ]:


# Create ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

# Fit and transform
X_traintrans= preprocessor.fit_transform(X_train)
X_testtrans =preprocessor.transform(X_test)



In [ ]:
X_traintrans.shape

In [ ]:
# Get names from OneHotEncoder (categorical)
cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)

# Numerical column names (same as original)
num_names = numerical_cols

# Combine them 
feature_names = list(cat_names) + list(num_names)

print(feature_names)
print(len(feature_names))
print(X_traintrans.shape)

In [ ]:
##BASELINE

In [ ]:
dummy=DummyClassifier(strategy='most_frequent')
dummy.fit(X_traintrans, y_train)





In [ ]:

print(dummy.score(X_traintrans, y_train))

In [ ]:
dummytrain_pred = dummy.predict(X_traintrans)
dummytest_pred = dummy.predict(X_testtrans)

In [ ]:
print("Dummy accuracy:", accuracy_score(y_train, dummytrain_pred))
print("Dummy accuracytest :", accuracy_score(y_test, dummytest_pred))

In [ ]:


from sklearn.metrics import precision_score, recall_score

precisiondummytest = precision_score(y_test, dummytest_pred, zero_division=0)
recalltestdummytest = recall_score(y_test, dummytest_pred, zero_division=0)
precisiondummytrain = precision_score(y_train, dummytrain_pred,zero_division=0)
recalldummytrain = recall_score(y_train, dummytrain_pred,zero_division=0)

In [ ]:
print("Precision test dummy:", precisiondummytest)
print("Recall test dummy :", recalltestdummytest)
print("Precision train dummy :", precisiondummytrain)
print("Recall train dummy :", recalldummytrain)

In [ ]:
print("Count of 0 in y_pred:", list(dummytrain_pred).count(0))
print("Count of 1 in y_pred:", list(dummytrain_pred).count(1))


## Machine Learning Models

This section trains different classification models and compares their performance.

The models used include:

- Logistic Regression  
- Random Forest  
- Support Vector Machine (SVM)  

The aim is to compare model behaviour using accuracy, precision and recall.


In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
#LOG REGRESSION
log_model = LogisticRegression()
log_model.fit(X_traintrans, y_train)

In [ ]:
print(log_model.score(X_traintrans, y_train))

In [ ]:



train_pred = log_model.predict(X_traintrans)
test_pred = log_model.predict(X_testtrans)



In [ ]:
print(" accuracy:", accuracy_score(y_train, train_pred))
print(" accuracytest :", accuracy_score(y_test, test_pred))

In [ ]:


print(classification_report(y_test, test_pred))

In [ ]:
#print(classification_report(y_train, train_pred))

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, test_pred))

In [ ]:
print("Count of 0 in y_pred:", list(train_pred).count(0))
print("Count of 1 in y_pred:", list(train_pred).count(1))


In [ ]:
precisiontest = precision_score(y_test, test_pred)
recalltest = recall_score(y_test, test_pred)
precisiontrain = precision_score(y_train, train_pred)
recalltrain = recall_score(y_train, train_pred)

In [ ]:
print("Precision test:", precisiontest)
print("Recall test:", recalltest)
print("Precision train:", precisiontrain)
print("Recall train:", recalltrain)

In [ ]:
##logreg tuning 
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
model_lr = LogisticRegression(max_iter=1000)
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

grid_lr = GridSearchCV(
    estimator=model_lr,
    param_grid=param_grid,
    scoring='recall',
    cv=5,
    n_jobs=-1
)



In [ ]:
grid_lr.fit(X_traintrans, y_train)

In [ ]:
print("Best parameters:", grid_lr.best_params_)
print("Best CV recall:", grid_lr.best_score_)

In [ ]:
best_lr = grid_lr.best_estimator_

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

y_pred_lr = best_lr.predict(X_testtrans)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))


## PCA and Dimensionality Reduction

Principal Component Analysis (PCA) is tested to reduce dimensionality and investigate whether combining features into components improves model performance.

PCA can sometimes improve recall by reducing noise, although it may reduce interpretability because the original feature meanings are transformed into components.


In [ ]:
#log regression pca

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

In [ ]:
pca = PCA(n_components=2)

X_train_pca = pca.fit_transform(X_traintrans)
X_test_pca = pca.transform(X_testtrans)
print("Explained variance:", pca.explained_variance_ratio_)

In [ ]:
model_lr_pca = LogisticRegression(max_iter=1000)
model_lr_pca.fit(X_train_pca, y_train)

In [ ]:
y_pred_lr_pca = model_lr_pca.predict(X_test_pca)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_lr_pca))
print("Precision:", precision_score(y_test, y_pred_lr_pca))
print("Recall:", recall_score(y_test, y_pred_lr_pca))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr_pca))

In [ ]:
for n in [2, 3, 4]:
    
    pca = PCA(n_components=n)
    X_train_pca = pca.fit_transform(X_traintrans)
    X_test_pca = pca.transform(X_testtrans)
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_pca, y_train)
    
    y_train_pred = model.predict(X_train_pca)
    y_test_pred = model.predict(X_test_pca)
    
    print(f"\nComponents: {n}")
    print("TRAIN Recall:", round(recall_score(y_train, y_train_pred), 3))
    print("TEST Recall:", round(recall_score(y_test, y_test_pred), 3))
    print("Accuracy:", round(accuracy_score(y_test, y_test_pred), 3))
    print("Precision:", round(precision_score(y_test, y_test_pred), 3))

In [ ]:
##SVM
from sklearn.svm import SVC

In [ ]:
model_svm = SVC(class_weight='balanced')

In [ ]:
model_svm.fit(X_traintrans, y_train)

In [ ]:
y_pred_svm = model_svm.predict(X_testtrans)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
print("TEST METRICS")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Precision:", precision_score(y_test, y_pred_svm))
print("Recall:", recall_score(y_test, y_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

y_train_pred = model_svm.predict(X_traintrans)

print("TRAIN METRICS")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("Precision:", precision_score(y_train, y_train_pred))
print("Recall:", recall_score(y_train, y_train_pred))
print("Confusion Matrix:\n", confusion_matrix(y_train, y_train_pred))

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.svm import LinearSVC
svm = LinearSVC(max_iter=5000, random_state=42)

param_grid_svm = {
    'C': [0.01, 0.1, 1, 10],
    'class_weight': [None, 'balanced']
}

grid_svm = GridSearchCV(
    svm,
    param_grid_svm,
    scoring='recall',
    cv=3,
    n_jobs=-1
)

grid_svm.fit(X_traintrans, y_train)

print("Best parameters:", grid_svm.best_params_)
print("Best CV recall:", grid_svm.best_score_)

In [ ]:
best_svm = grid_svm.best_estimator_

# Train predictions
y_train_pred_svm = best_svm.predict(X_traintrans)

# Test predictions
y_test_pred_svm = best_svm.predict(X_testtrans)

print("TRAIN")
print("Accuracy:", accuracy_score(y_train, y_train_pred_svm))
print("Precision:", precision_score(y_train, y_train_pred_svm))
print("Recall:", recall_score(y_train, y_train_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_train, y_train_pred_svm))

print("\nTEST")
print("Accuracy:", accuracy_score(y_test, y_test_pred_svm))
print("Precision:", precision_score(y_test, y_test_pred_svm))
print("Recall:", recall_score(y_test, y_test_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred_svm))

In [ ]:
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score

for n in [2, 3, 4]:
    
    # Apply PCA
    pca = PCA(n_components=n)
    X_train_pca = pca.fit_transform(X_traintrans)
    X_test_pca = pca.transform(X_testtrans)
    
    # Train SVM
    model = SVC()
    model.fit(X_train_pca, y_train)
    
    # Predict
    y_pred = model.predict(X_test_pca)
    
    # Print results
    print(f"\nComponents: {n}")
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
    print("Precision:", round(precision_score(y_test, y_pred), 3))
    print("Recall:", round(recall_score(y_test, y_pred), 3))

In [ ]:
###random forrest
features_rf = ['leverage','cash_ratio','ppe_ratio','retained_ratio','size']
target = 'high_performance'

X_rf = df[features_rf]
y = df[target]

In [ ]:
from sklearn.model_selection import train_test_split

X_train_rf, X_test_rf, y_train, y_test = train_test_split(
    X_rf, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
num_cols = X_train_rf.select_dtypes(include='number').columns
median = X_train_rf[num_cols].median()

X_train_rf[num_cols] = X_train_rf[num_cols].fillna(median)
X_test_rf[num_cols] = X_test_rf[num_cols].fillna(median)

In [ ]:
X_train_rf['size'] = X_train_rf['size'].astype(str).replace('nan','Unknown')
X_test_rf['size'] = X_test_rf['size'].astype(str).replace('nan','Unknown')

In [ ]:
X_train_rf = pd.get_dummies(X_train_rf, columns=['size'], drop_first=True)
X_test_rf = pd.get_dummies(X_test_rf, columns=['size'], drop_first=True)

In [ ]:
X_train_rf, X_test_rf = X_train_rf.align(X_test_rf, join='left', axis=1, fill_value=0)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train_rf, y_train)

In [ ]:
y_pred_rf = model_rf.predict(X_test_rf)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
print("Train Accuracy rf")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

In [ ]:
y_train_pred_rf = model_rf.predict(X_train_rf)

print("Train Accuracy:", accuracy_score(y_train, y_train_pred_rf))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# Random Forest base model
rf = RandomForestClassifier(random_state=42)

# Tuning grid to reduce overfitting
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 8, 10],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [3, 5, 10],
    'max_features': ['sqrt', 'log2']
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='recall',   # recall is your priority
    cv=5,
    n_jobs=-1
)

# Fit only on training data
grid_rf.fit(X_train_rf, y_train)

print("Best parameters:", grid_rf.best_params_)
print("Best CV recall:", grid_rf.best_score_)

In [ ]:
best_rf = grid_rf.best_estimator_

# Train performance
y_train_pred = best_rf.predict(X_train_rf)

# Test performance
y_test_pred = best_rf.predict(X_test_rf)

print("TRAIN")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("Precision:", precision_score(y_train, y_train_pred))
print("Recall:", recall_score(y_train, y_train_pred))
print("Confusion Matrix:\n", confusion_matrix(y_train, y_train_pred))

print("\nTEST")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))

In [ ]:
print(grid_rf.best_params_)

In [ ]:

acc_lr=0.74
prec_lr=0.66
rec_lr=0.931
acc_rf=0.83
prec_rf=0.82
rec_rf=0.82
acc_svm=0.72
prec_svm =0.63
rec_svm=0.928


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Create results table
results = pd.DataFrame({
    'Model': ['Logistic Regression PCA', 'Random Forest', 'SVM'],
    'Accuracy': [acc_lr, acc_rf, acc_svm],
    'Precision': [prec_lr, prec_rf, prec_svm],
    'Recall': [rec_lr, rec_rf, rec_svm]
})

# Set model names as index
results.set_index('Model', inplace=True)

# Plot
results.plot(kind='bar', figsize=(10,6))

plt.ylabel('Score')
plt.title('Model Performance Comparison')
plt.xticks(rotation=0)

plt.show()

Logistic Regression with PCA was selected as the final model because it achieved the highest recall while maintaining acceptable accuracy and interpretability. Since the project prioritized minimizing false negatives and identifying as many high-performing firms as possible, recall was considered the most important evaluation metric.


## Business Interpretation

This section connects model errors to business consequences.

In this project, false negatives are especially important because they represent high-performing firms that the model failed to identify.

A missed high-performing firm can be interpreted as a missed investment opportunity, which supports the decision to prioritise recall during model selection.


In [ ]:
top_roa = dforiginal[dforiginal['high_performance'] == 1]['ROA'].mean()

# Average ROA for low-performing firms
bottom_roa = dforiginal[dforiginal['high_performance'] == 0]['ROA'].mean()

# Difference (opportunity gap)
roa_gap = top_roa - bottom_roa

print("Average ROA - High-performing firms:", round(top_roa * 100, 2), "%")
print("Average ROA - Low-performing firms:", round(bottom_roa * 100, 2), "%")
print("ROA Opportunity Gap:", round(roa_gap * 100, 2), "%")

In [ ]:
# Get encoded categorical column names
cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)

# Numerical column names
num_names = numerical_cols

# Combine names in correct order
feature_names = list(cat_names) + list(num_names)

print(feature_names)


## Feature Interpretation

This section examines which features have the strongest influence on predictions.

Logistic Regression coefficients help explain the direction of impact:

- positive coefficients increase the probability of high performance  
- negative coefficients decrease the probability of high performance  

Random Forest feature importance can also be used to understand which variables are most useful for prediction.


In [ ]:


## log reg features impact 
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': best_lr.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print(coef_df)

In [ ]:
## random forrest features impact 
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': model_rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importance_df)

In [ ]:
dforiginal.head()

In [ ]:
# Remove extreme ROA outliers (1st and 99th percentile)
filtered_roa = dforiginal[
    (dforiginal['ROA'] > dforiginal['ROA'].quantile(0.01)) &
    (dforiginal['ROA'] < dforiginal['ROA'].quantile(0.99))
]

# Median ROA for high-performing firms
top_roa = filtered_roa[
    filtered_roa['high_performance'] == 1
]['ROA'].median()

# Median ROA for low-performing firms
bottom_roa = filtered_roa[
    filtered_roa['high_performance'] == 0
]['ROA'].median()

# Opportunity gap
roa_gap = top_roa - bottom_roa

print("Median ROA - High-performing firms:", round(top_roa * 100, 2), "%")
print("Median ROA - Low-performing firms:", round(bottom_roa * 100, 2), "%")
print("ROA Opportunity Gap:", round(roa_gap * 100, 2), "%")

Median ROA analysis showed a substantial performance gap between high-performing and low-performing firms. High-performing firms exhibited a median ROA of 2.88%, while low-performing firms showed a median ROA of -55.7%, resulting in an opportunity gap of approximately 58.6%. This highlights the significant financial cost associated with misclassification.

Based on median ROA differences between performance groups, missing a truly high-performing company may correspond to an opportunity cost of approximately 59%, while incorrectly selecting a low-performing firm may expose investors to approximately 56% underperformance.

In [ ]:
plt.hist(dforiginal['ROA'], bins=30)

plt.xlabel('ROA')
plt.ylabel('Frequency')
plt.title('Distribution of Original ROA')

plt.show()

In [ ]:
dforiginal['ROA'].describe

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

roa_clean = dforiginal['ROA'].replace([np.inf, -np.inf], np.nan).dropna()

# Keep middle 98% only for visualization
roa_plot = roa_clean[
    (roa_clean >= roa_clean.quantile(0.01)) &
    (roa_clean <= roa_clean.quantile(0.99))
]

plt.figure(figsize=(8,5))
plt.hist(roa_plot, bins=50, edgecolor='black')

plt.xlabel('ROA')
plt.ylabel('Number of Companies')
plt.title('Distribution of ROA - Outliers Removed for Visualization')

plt.show()

In [ ]:
print(dforiginal['ROA'].describe())



print(dforiginal['ROA'].min())
print(dforiginal['ROA'].max())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

roa_clean = dforiginal['ROA'].replace([np.inf, -np.inf], np.nan).dropna()

roa_plot = roa_clean[
    (roa_clean >= roa_clean.quantile(0.01)) &
    (roa_clean <= roa_clean.quantile(0.99))
]

plt.figure(figsize=(10,5))

plt.hist(roa_plot, bins=100, edgecolor='black')

plt.xlabel('ROA')
plt.ylabel('Frequency')
plt.title('Distribution of ROA')

plt.show()

In [ ]:
features = ['leverage', 'cash_ratio', 'ppe_ratio']

corr_with_roa = dforiginal[features + ['ROA']].corr()['ROA']

print(corr_with_roa.sort_values(ascending=False))

Correlation analysis showed that leverage exhibited the strongest relationship with ROA, with higher leverage associated with lower profitability. PPE ratio and cash ratio displayed limited direct linear relationships with ROA.
Simple linear relationships alone do not fully explain firm performance.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Clean ROA
roa_clean = dforiginal['ROA'].replace([np.inf, -np.inf], np.nan).dropna()

# Plot histogram
plt.figure(figsize=(10,5))

plt.hist(roa_clean, bins=100)

# Log scale on y-axis
plt.yscale('log')

plt.xlabel('ROA')
plt.ylabel('Log Frequency')
plt.title('Distribution of ROA (Log Scale)')

plt.show()

In [ ]:
roa_plot = roa_clean[
    (roa_clean >= roa_clean.quantile(0.01)) &
    (roa_clean <= roa_clean.quantile(0.99))
]

In [ ]:
plt.boxplot(roa_plot)

In [ ]:
plt.hist(roa_plot, bins=50)

In [ ]:

import pandas as pd

results = pd.DataFrame({
    'Model': [
        'Logistic + PCA',
        'Random Forest',
        'SVM'
    ],

    'Accuracy': [
        0.74,
        0.83,
        0.72
    ],

    'Precision': [
        0.66,
        0.82,
        0.63
    ],

    'Recall': [
        0.931,
        0.82,
        0.928
    ]
})

results.to_csv("model_results.csv", index=False)


In [ ]:
import pandas as pd

coef_table = pd.DataFrame({
    'Feature': [
        'size_Large',
        'size_Medium',
        'size_Small',
        'size_Unknown',
        'cash_ratio',
        'leverage',
        'ppe_ratio'
    ],

    'Coefficient': [
        2.612589,
        1.224023,
        -0.840411,
        -4.258562,
        -0.224768,
        -0.075550,
        -0.038332
    ]
})

coef_table.to_csv("feature_importance.csv", index=False)

In [ ]:
coef_table.head()

In [ ]:
coef_table.to_csv("feature_importance.csv", index=False)

## Final Conclusion

The project showed that financial ratios and company characteristics can be used to classify firm performance. Company size was the strongest explanatory variable, while cash ratio, leverage and PPE ratio contributed additional financial insight.

Logistic Regression with PCA was selected as the final model because it achieved the highest recall while maintaining interpretability. This matched the business objective of reducing missed high-performing firms.